# 06 · Cellular Automata (Rules 232 & 30) — learn the mapping


> **TL;DR**  
> This notebook runs a *paired* comparison between **Boolearn** (C, RRR constraints) and PyTorch models trained with **AdamW**, **SGD+Momentum**, and **Muon** (if installed).  
> We measure **generalization** under two modes:
>
> 1) **Black‑box**: flexible MLP + small hyper‑parameter sweep; split the hold‑out into **val**/**test** and pick the best by val.  
> 2) **Principled**: architecture/activations closer to Boolean circuits + L1 to promote **sparsity**; minimal tuning.
>
> **How few training examples** are needed to reach **perfect test accuracy**?
>
> ---


In [10]:
from pathlib import Path
import subprocess, os

# ==============================
# OPTIONAL: set this if Jupyter is launched outside the repo
# (recommended default for robustness)
# ==============================
REPO_HINT = Path("/Users/mkrishan/Documents/boolearn-main")
# Users can comment this out if they run notebooks from inside the repo


def find_repo_root():
    candidates = []

    # 1) user-provided hint
    if REPO_HINT is not None:
        candidates.append(Path(REPO_HINT).expanduser().resolve())

    # 2) upward search from current working directory
    cwd = Path.cwd().resolve()
    candidates += [cwd] + list(cwd.parents)

    # 3) common fallbacks
    home = Path.home()
    candidates += [
        home / "boolearn-main",
        home / "Documents" / "boolearn-main",
        home / "Desktop" / "boolearn-main",
    ]

    # valid repo = boolearn/src/makefile exists
    for p in candidates:
        if (p / "boolearn" / "src" / "makefile").exists():
            return p

    raise RuntimeError(
        "❌ Could not locate Boolearn repository root.\n"
        "Fix: set REPO_HINT = Path('.../boolearn-main') at the top of this cell.\n"
        f"Current working directory: {cwd}"
    )


# ==============================
# Locate repo + key paths
# ==============================
ROOT = find_repo_root()
SRC  = ROOT / "boolearn" / "src"
DATA = ROOT / "data"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("✅ Repo root:", ROOT)
print("📁 SRC:", SRC)
print("📁 DATA:", DATA)
print("📁 RESULTS:", RESULTS)

# ==============================
# Build boolearn binaries
# ==============================
subprocess.run(["make", "all"], cwd=str(SRC), check=True)

# ==============================
# Add boolearn/src to PATH
# ==============================
os.environ["PATH"] = f"{SRC}:{os.environ.get('PATH','')}"
print("✅ PATH updated with boolearn/src")

✅ Repo root: /Users/mkrishan/Documents/boolearn-main
📁 SRC: /Users/mkrishan/Documents/boolearn-main/boolearn/src
📁 DATA: /Users/mkrishan/Documents/boolearn-main/data
📁 RESULTS: /Users/mkrishan/Documents/boolearn-main/results
make: Nothing to be done for `all'.
✅ PATH updated with boolearn/src


In [18]:
from pathlib import Path

# Ensure repo paths exist (Cell 0 must have run)
assert "SRC" in globals(), "Run Cell 0 (repo setup) first."

# Resolve boolearn executables explicitly
data_cellauto = SRC / "data_cellauto"
train        = SRC / "train"
layered      = SRC / "layered"

assert data_cellauto.exists(), f"Missing {data_cellauto}"
assert train.exists(),        f"Missing {train}"
assert layered.exists(),      f"Missing {layered}"

print("✅ data_cellauto:", data_cellauto)
print("✅ train:", train)
print("✅ layered:", layered)

✅ data_cellauto: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto
✅ train: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train
✅ layered: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered


In [20]:
import subprocess
from pathlib import Path

# Make sure prior cells ran
assert "DATA" in globals() and "SRC" in globals(), "Run Cell 0 first."
assert "data_cellauto" in globals(), "Run the executable-binding cell first."

cellauto_dir = DATA / "cellauto"
cellauto_dir.mkdir(parents=True, exist_ok=True)

ID = "30_1"
DAT = cellauto_dir / f"{ID}.dat"

INBITS = 3
RULE   = 30
SIZE   = 16
STEPS  = 1
ITEMS  = 10000

if not DAT.exists():
    cmd = [str(data_cellauto), str(INBITS), str(RULE), str(SIZE), str(STEPS), str(ITEMS), ID]
    print("Generating:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cellauto_dir), check=True)
else:
    print("Found:", DAT)

# quick header check
with open(DAT) as f:
    for _ in range(6):
        print(f.readline().rstrip())

Generating: /Users/mkrishan/Documents/boolearn-main/boolearn/src/data_cellauto 3 30 16 1 10000 30_1
10000
2 16
2 16

 0 0 1 0 0 0 1 0 0 1 1 0 0 1 1 1
 1 1 1 1 0 1 1 1 1 1 0 1 1 1 0 0


In [22]:
import subprocess
from pathlib import Path

# Preconditions
assert "cellauto_dir" in globals(), "Run previous cell first."
assert "layered" in globals(), "Executable 'layered' not defined."

ID = "30_1"
DAT = cellauto_dir / f"{ID}.dat"
assert DAT.exists(), f"Missing dataset: {DAT}"

NET_DIR = RESULTS / "cellauto_sanity" / "nets"
NET_DIR.mkdir(parents=True, exist_ok=True)

# Write width file (wth)
WTH = NET_DIR / f"{ID}.wth"
NET = NET_DIR / f"{ID}.net"

if not WTH.exists():
    with open(WTH, "w") as f:
        f.write("3\n")
        f.write("32 32\n")   # two hidden layers
    print("Wrote:", WTH)
else:
    print("Found:", WTH)

# Build net using layered
if not NET.exists():
    cmd = [str(layered), "0.5", str(WTH), str(NET)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=str(NET_DIR), check=False)
else:
    print("Found:", NET)

# Validate net
assert NET.exists(), "❌ layered did not produce a .net file"

print("✅ Using net:", NET, "| bytes:", NET.stat().st_size)

with open(NET) as f:
    header = f.readline().strip()
print("Header:", header)

Wrote: /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.wth
Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered 0.5 /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.wth /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.net
✅ Using net: /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.net | bytes: 26416
Header: 65  33  0  1056


In [24]:
import subprocess
from pathlib import Path

ID = "30_1"

NET_DIR = RESULTS / "cellauto_sanity" / "nets"
NET_DIR.mkdir(parents=True, exist_ok=True)

WTH = NET_DIR / f"{ID}.wth"
NET = NET_DIR / f"{ID}.net"

# Correct width file: L then "in h1 h2 out"
with open(WTH, "w") as f:
    f.write("3\n")
    f.write("16 32 32 16\n")

print("✅ Rewrote:", WTH)
print(WTH.read_text())

# Rebuild net (overwrite)
if NET.exists():
    NET.unlink()

cmd = [str(layered), "0.5", str(WTH), str(NET)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, cwd=str(NET_DIR), check=False)

assert NET.exists() and NET.stat().st_size > 0
print("✅ Net:", NET, "| bytes:", NET.stat().st_size)

with open(NET) as f:
    header = f.readline().strip()
print("Header:", header)

✅ Rewrote: /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.wth
3
16 32 32 16

Running: /Users/mkrishan/Documents/boolearn-main/boolearn/src/layered 0.5 /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.wth /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.net
✅ Net: /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.net | bytes: 53217
Header: 97  17  16  2128


In [26]:
import subprocess, time, re, select
from pathlib import Path

ID = "30_1"
DAT = cellauto_dir / f"{ID}.dat"
NET = RESULTS / "cellauto_sanity" / "nets" / f"{ID}.net"
assert DAT.exists() and NET.exists()

RUN_DIR = RESULTS / "cellauto_sanity" / "btf"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# sanity N (change later to [64,96,128,160] etc.)
N_train = 256

# short run id
run_id = f"r30N{N_train}"
gap_path = RUN_DIR / f"{run_id}.gap"

# hyperparams (sanity)
btfn   = "3.0"
beta   = "0.2"
gam    = "1e-3"
epochs = "80"
itermax= "50000"   # sanity
gapstop= "0.1"
trials = "1"

cmd = [
    str(train),
    str(NET),
    str(DAT),
    str(N_train),
    btfn, beta, gam, epochs, itermax, gapstop, trials,
    run_id
]

print("Running boolearn:", " ".join(cmd))
print("RUN_DIR:", RUN_DIR)
print("Watching:", gap_path.name)

proc = subprocess.Popen(
    cmd,
    cwd=str(RUN_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

last_gap_lines = 0
printed_header = False

try:
    while True:
        # discard stdout without blocking
        if proc.stdout is not None:
            r, _, _ = select.select([proc.stdout], [], [], 0.0)
            if r:
                _ = proc.stdout.readline()

        # print only new gap lines (log steps)
        if gap_path.exists():
            lines = [ln.strip() for ln in gap_path.read_text().splitlines() if ln.strip()]

            if lines and not printed_header:
                print("\niter\ttrain%\ttest%")
                printed_header = True

            if len(lines) > last_gap_lines:
                for ln in lines[last_gap_lines:]:
                    it = ln.split()[0]
                    pcts = re.findall(r"(\d+(?:\.\d+)?)\s*%", ln)
                    if len(pcts) >= 2:
                        tr, te = float(pcts[-2]), float(pcts[-1])
                        print(f"{it}\t{tr:.3f}\t{te:.3f}")
                last_gap_lines = len(lines)

        if proc.poll() is not None:
            break

        time.sleep(0.2)

finally:
    try:
        proc.kill()
    except Exception:
        pass

print("\nReturn code:", proc.returncode)

print("\nArtifacts:")
for p in sorted(RUN_DIR.glob(run_id + ".*")):
    print(" -", p.name, "|", p.stat().st_size, "bytes")

Running boolearn: /Users/mkrishan/Documents/boolearn-main/boolearn/src/train /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/nets/30_1.net /Users/mkrishan/Documents/boolearn-main/data/cellauto/30_1.dat 256 3.0 0.2 1e-3 80 50000 0.1 1 r30N256
RUN_DIR: /Users/mkrishan/Documents/boolearn-main/results/cellauto_sanity/btf
Watching: r30N256.gap

iter	train%	test%
2	47.729	48.114
3	48.169	48.282
4	48.169	48.232
5	48.193	48.239
6	48.242	48.269
7	48.120	48.284
8	48.633	48.269
9	48.218	48.243
10	48.364	48.262
11	48.291	48.247
12	48.462	48.222
13	48.315	48.230
14	48.560	48.249
15	48.486	48.232
16	48.657	48.231
17	48.560	48.264
18	48.657	48.221
19	48.779	48.237
20	48.584	48.259
21	48.291	48.286
22	48.608	48.264
23	48.315	48.255
24	48.486	48.344
26	48.438	48.345
30	48.682	48.435
34	49.072	48.481
39	48.853	48.565
45	50.024	48.813
51	50.024	49.169
58	51.343	49.577
67	51.514	50.065
76	51.392	50.724
87	52.856	51.647
100	54.346	52.598
114	55.273	53.594
131	56.934	54.531
150	58.105	55.487

In [27]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# --- paths ---
ID = "30_1"
DAT = cellauto_dir / f"{ID}.dat"
assert DAT.exists(), DAT

GD_DIR = RESULTS / "cellauto_sanity" / "gd"
GD_DIR.mkdir(parents=True, exist_ok=True)

# --- load boolearn .dat (binary) ---
def load_dat_bits(path: Path):
    with open(path) as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    it = iter(lines)
    N_total = int(next(it))
    Tin, IN = map(int, next(it).split())
    Tout, OUT = map(int, next(it).split())
    X, Y = [], []
    for _ in range(N_total):
        X.append(list(map(int, next(it).split())))
        Y.append(list(map(int, next(it).split())))
    X = torch.tensor(X, dtype=torch.float32)
    Y = torch.tensor(Y, dtype=torch.float32)
    return X, Y, IN, OUT

# --- log-spaced checkpoints (dots) ---
def log_spaced_steps(max_steps: int, n_points: int = 35):
    steps = np.unique(np.round(np.logspace(0, np.log10(max_steps), n_points)).astype(int))
    steps = steps[(steps >= 1) & (steps <= max_steps)]
    return set([0] + steps.tolist())

@torch.no_grad()
def bitwise_and_exact_acc(logits, y_true, thresh=0.5):
    probs = torch.sigmoid(logits)
    pred = (probs >= thresh).float()
    bit_acc = (pred == y_true).float().mean().item() * 100.0
    exact_acc = (pred.eq(y_true).all(dim=1)).float().mean().item() * 100.0
    return bit_acc, exact_acc

# --- GD model: 16 -> 32 -> 32 -> 16 ---
class MLP16_32_32_16(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(16, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 32), nn.ReLU(inplace=True),
            nn.Linear(32, 16),
        )
    def forward(self, x):
        return self.net(x)

# --- load data ---
X_all, Y_all, IN, OUT = load_dat_bits(DAT)
assert IN == 16 and OUT == 16
assert len(X_all) == 10000

# --- split ---
N_train = 256
Xtr = X_all[:N_train].to(device)
Ytr = Y_all[:N_train].to(device)
Xte = X_all[N_train:].to(device)
Yte = Y_all[N_train:].to(device)

# --- run config (sanity) ---
MAX_STEPS = 20000
LOG_STEPS = log_spaced_steps(MAX_STEPS, n_points=35)

SEED = 0
torch.manual_seed(SEED)

model = MLP16_32_32_16().to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

gd_log = GD_DIR / f"r30N{N_train}_adamw_sanity.log"
with open(gd_log, "w") as f:
    f.write("# step,train_loss,test_loss,train_bitacc,test_bitacc,train_exactacc,test_exactacc\n")

def eval_and_log(step: int):
    model.eval()
    with torch.no_grad():
        tr_logits = model(Xtr)
        te_logits = model(Xte)
        tr_loss = loss_fn(tr_logits, Ytr).item()
        te_loss = loss_fn(te_logits, Yte).item()
        tr_bit, tr_ex = bitwise_and_exact_acc(tr_logits, Ytr)
        te_bit, te_ex = bitwise_and_exact_acc(te_logits, Yte)

    with open(gd_log, "a") as f:
        f.write(f"{step},{tr_loss:.6f},{te_loss:.6f},{tr_bit:.4f},{te_bit:.4f},{tr_ex:.4f},{te_ex:.4f}\n")

    print(f"[GD r30 N={N_train}] step={step:>6d}  "
          f"train={tr_bit:6.2f}% test={te_bit:6.2f}%  exact={tr_ex:5.1f}/{te_ex:5.1f}")

# step 0
eval_and_log(0)

model.train()
for step in range(1, MAX_STEPS + 1):
    opt.zero_grad(set_to_none=True)
    loss = loss_fn(model(Xtr), Ytr)
    loss.backward()
    opt.step()

    if step in LOG_STEPS:
        eval_and_log(step)

print("✅ Saved:", gd_log)

device: cpu
[GD r30 N=256] step=     0  train= 49.93% test= 49.38%  exact=  0.0/  0.0
[GD r30 N=256] step=     1  train= 50.12% test= 49.46%  exact=  0.0/  0.0
[GD r30 N=256] step=     2  train= 50.29% test= 49.60%  exact=  0.0/  0.0
[GD r30 N=256] step=     3  train= 50.51% test= 49.75%  exact=  0.0/  0.0
[GD r30 N=256] step=     4  train= 50.93% test= 49.91%  exact=  0.0/  0.0
[GD r30 N=256] step=     6  train= 51.34% test= 50.24%  exact=  0.0/  0.0
[GD r30 N=256] step=     8  train= 51.61% test= 50.58%  exact=  0.0/  0.0
[GD r30 N=256] step=    10  train= 52.42% test= 50.97%  exact=  0.4/  0.0
[GD r30 N=256] step=    14  train= 53.93% test= 51.72%  exact=  0.4/  0.0
[GD r30 N=256] step=    18  train= 55.15% test= 52.48%  exact=  0.4/  0.0
[GD r30 N=256] step=    25  train= 57.91% test= 54.17%  exact=  0.4/  0.0
[GD r30 N=256] step=    33  train= 60.42% test= 55.68%  exact=  0.0/  0.0
[GD r30 N=256] step=    44  train= 61.89% test= 56.61%  exact=  0.0/  0.0
[GD r30 N=256] step=    59